<a href="https://colab.research.google.com/github/WalterPerjenko/LW_Statistic/blob/main/%D0%9B%D0%B0%D0%B1%D0%B0_%D1%81%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Статистическое исследование данных AqSolDB

<a id="contents"></a>

## Навигация по исследованию

1. [Постановка задачи, данные и подготовка среды](#section-1)
2. [Очистка и формирование генеральной совокупности](#section-2)
3. [Описательные статистики](#section-3)
4. [Распределения и крайние наблюдения](#section-4)
5. [Стандартизация и нормализация](#section-5)
6. [Генеральные и выборочные характеристики](#section-6)
7. [Сходимость статистических оценок](#section-7)
8. [Выводы по основному исследованию](#section-8)
9. [Дополнительный анализ признака MolLogP](#section-9)

---

**Объект исследования:** очищенная база химических соединений AqSolDB.  
**Основные признаки:** `LogS` и `MolWt`.  
**Дополнительный анализ:** `MolLogP`, растворимость и корреляционные связи.

В работе рассматриваются описание генеральной совокупности, распределения признаков, преобразование шкал, моделирование выборочного отбора и поведение статистических оценок при увеличении объёма выборки.

[К оглавлению](#contents)


<a id="section-1"></a>
## 1. Постановка задачи, данные и подготовка среды

На этом этапе подключаются библиотеки, задаются анализируемые признаки и параметры воспроизводимости, загружается исходный набор данных и сохраняется его исходная копия.

[К оглавлению](#contents)

In [ ]:
#@title 1. Настройки и загрузка данных {display-mode: "form"}
from pathlib import Path
import io
import json
import hashlib
import zipfile
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from IPython.display import display, Markdown


# 1. НАСТРОЙКИ И ЗАГРУЗКА ДАННЫХ

SEED = 42

# признаки (Даша)
FEATURES = ["LogS", "MolWt"]

# все признаки
ALL_FEATURES = ["LogS", "MolWt", "MolLogP"]

URL = (
    "https://raw.githubusercontent.com/"
    "mcsorkun/AqSolDB/master/results/data_curated.csv"
)

OUT = Path("aqsoldb_results")
OUT.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font="DejaVu Sans")

# Единая стилистика графиков исследования
PLOT_COLORS = {
    "primary": "#3366A6",
    "secondary": "#E07A3F",
    "tertiary": "#3A8D75",
    "neutral": "#555555",
    "grid": "#D9DEE7",
}
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette=[PLOT_COLORS["primary"], PLOT_COLORS["secondary"],
             PLOT_COLORS["tertiary"], PLOT_COLORS["neutral"]],
    font="DejaVu Sans",
    rc={
        "figure.figsize": (10, 6),
        "figure.dpi": 110,
        "savefig.dpi": 180,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "axes.edgecolor": "#AAB2BF",
        "axes.linewidth": 0.8,
        "grid.color": PLOT_COLORS["grid"],
        "grid.alpha": 0.7,
        "grid.linewidth": 0.7,
        "axes.spines.top": False,
        "axes.spines.right": False,
    },
)


with urlopen(URL, timeout=60) as response:
    source_bytes = response.read()

raw = pd.read_csv(io.BytesIO(source_bytes))

# сохранение исходных данныз
(OUT / "aqsoldb_raw.csv").write_bytes(source_bytes)

(OUT / "source.json").write_text(
    json.dumps(
        {
            "url": URL,
            "sha256": hashlib.sha256(source_bytes).hexdigest(),
            "seed": SEED,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Исходный размер:", raw.shape)
display(raw.head())

display(
    pd.DataFrame(
        {
            "Тип": raw.dtypes.astype(str),
            "Пропусков": raw.isna().sum(), # количество отсутствующих значений в столбце.
        }
    )
)



<a id="section-2"></a>
## 2. Очистка и формирование генеральной совокупности

Проводится переименование признака растворимости, удаление полных дубликатов, приведение исследуемых полей к числовому типу и фильтрация некорректных наблюдений. Итоговая очищенная таблица рассматривается в работе как генеральная совокупность.

[К оглавлению](#contents)

In [ ]:
#@title 2. Очистка данных {display-mode: "form"}


# 2. ОЧИСТКА ОБЩЕЙ ТАБЛИЦЫ


# переименование , повторный логарифм не берётся.
data = raw.rename(columns={"Solubility": "LogS"}).copy()

duplicate_count = int(data.duplicated().sum())
data = data.drop_duplicates().copy()

# к числовуму типу
for c in ALL_FEATURES:
    data[c] = pd.to_numeric(data[c], errors="coerce")

display(
    pd.DataFrame(
        {
            "Пропуски / нечисловые": data[ALL_FEATURES].isna().sum(),
            "Бесконечности": np.isinf(data[ALL_FEATURES]).sum(),
        }
    )
)

# положительная молярная
valid = (
    np.isfinite(data[ALL_FEATURES]).all(axis=1)
    & (data["MolWt"] > 0)
)

invalid_count = int((~valid).sum())
data = data.loc[valid].reset_index(drop=True)

# проверка повторов  по InChIKey (если отдельные числовые свойства совпадают то это не дубликат)
keys = data["InChIKey"].astype("string").str.strip()

repeated = (
    keys.notna()
    & keys.ne("")
    & keys.duplicated(keep=False)
)

if repeated.any():
    display(data.loc[repeated, ["InChIKey"] + ALL_FEATURES])
    raise ValueError(
        "Повторные соединения: требуется проверить InChIKey."
    )

N = len(data)

if N < 10 or (data[FEATURES].nunique() < 2).any():
    raise ValueError(
        "Недостаточно корректных неодинаковых наблюдений."
    )

data.to_csv(OUT / "aqsoldb_clean.csv", index=False)

print(
    f"Полных повторов: {duplicate_count}; "
    f"некорректных строк: {invalid_count}; "
    f"N = {N}."
)

display(
    Markdown(
        r"""
**Случайный эксперимент:** равновероятно выбрать соединение
из очищенной базы.

LogS = log₁₀(S / (1 моль/л)),  S растворимость
MolWt = молярная масса, г/моль
MolLogP =  рассчитанный логарифм коэффициента распределения октанол/вода
каждая строка имее вероятность 1/N.
генеральная совокупность данного исследования = очищенный AqSolDB.

"""
    )
)

<a id="section-3"></a>
## 3. Описательные статистики

Для `LogS` и `MolWt` рассчитываются характеристики положения, разброса и формы распределения. Генеральная дисперсия рассчитывается с делителем \(N\):

\[
\mu = \frac{1}{N}\sum_{i=1}^{N}x_i,\qquad
\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2,\qquad
\sigma = \sqrt{\sigma^2}.
\]

[К оглавлению](#contents)

In [ ]:
#@title 3. Описательные статистики {display-mode: "form"}
# 3. ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ LogS И MolWt

population = data[FEATURES]

# ген среднее
mu = population.mean()

# ген дисперсия: делитель N
variance = population.var(ddof=0)

# ген стандартное отклонение
sigma = np.sqrt(variance)

summary = pd.DataFrame({
    "N": population.count(),
    "Среднее": mu,
    "Медиана": population.median(),
    "Дисперсия (N)": variance,
    "Стандартное отклонение": sigma,
    "Минимум": population.min(),
    "Q1": population.quantile(0.25),
    "Q3": population.quantile(0.75),
    "Максимум": population.max(),
    "Асимметрия": [
        skew(population[c], bias=True)
        for c in FEATURES
    ],
    "Эксцесс": [
        kurtosis(population[c], fisher=True, bias=True)
        for c in FEATURES
    ],
})

display(summary.round(5))

summary.to_csv(OUT / "population_statistics.csv")

<a id="section-4"></a>
## 4. Распределения и крайние наблюдения

Распределения исследуются с помощью гистограмм плотности, boxplot и эмпирической функции распределения. Крайние наблюдения отмечаются по правилу межквартильного размаха:

\[
IQR = Q_3-Q_1,\qquad
x < Q_1-1.5\,IQR\quad \text{или}\quad x > Q_3+1.5\,IQR.
\]

Отмеченные наблюдения не удаляются из данных.

[К оглавлению](#contents)

In [ ]:
#@title 4. Распределения и крайние наблюдения {display-mode: "form"}
# 4. РАСПРЕДЕЛЕНИЯ И КРАЙНИЕ НАБЛЮДЕНИЯ

labels = {
    "LogS": "LogS = log₁₀(S / (1 моль/л))",
    "MolWt": "Молярная масса, г/моль",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
outlier_rows = []

for row, c in enumerate(FEATURES):
    x = data[c]

    # границы по правилу межквартильного размаха
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    flagged = (x < lower) | (x > upper)

    outlier_rows.append({
        "Признак": c,
        "Нижняя граница": lower,
        "Верхняя граница": upper,
        "Отмечено": int(flagged.sum()),
        "Доля, %": 100 * flagged.mean(),
    })

    # гстограмма
    sns.histplot(
        x=x, bins="fd", stat="density", ax=axes[row, 0]
    )
    axes[row, 0].axvline(
        mu[c], color=PLOT_COLORS["secondary"], label="Среднее"
    )
    axes[row, 0].axvline(
        x.median(), color=PLOT_COLORS["neutral"],
        linestyle="--", label="Медиана"
    )
    axes[row, 0].set(
        xlabel=labels[c],
        ylabel="Плотность",
        title=f"{c}: гистограмма",
    )
    axes[row, 0].legend()

    # боксплот
    sns.boxplot(x=x, ax=axes[row, 1])
    axes[row, 1].set(
        xlabel=labels[c],
        title=f"{c}: boxplot",
    )

    # Эмпирическая функция распределения
    sns.ecdfplot(x=x, ax=axes[row, 2])
    axes[row, 2].set(
        xlabel=labels[c],
        ylabel="Fₙ(t)",
        title=f"{c}: ECDF",
    )

plt.tight_layout()
plt.savefig(
    OUT / "01_distributions.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

outliers = pd.DataFrame(outlier_rows).set_index("Признак")

display(outliers.round(4))
outliers.to_csv(OUT / "iqr_flags.csv")


<a id="section-5"></a>
## 5. Стандартизация и нормализация

Для каждого основного признака рассчитываются стандартизованное значение и min–max-преобразование:

\[
z_i = \frac{x_i-\mu}{\sigma},\qquad
u_i = \frac{x_i-x_{\min}}{x_{\max}-x_{\min}}.
\]

Отдельно рассматривается логарифмирование молярной массы. Единицы исходной величины `MolWt` — г/моль; после стандартизации и min–max-нормализации значения безразмерны.

[К оглавлению](#contents)

In [ ]:
#@title 5. Стандартизация и нормализация {display-mode: "form"}

# 5. СТАНДАРТИЗАЦИЯ И НОРМАЛИЗАЦИЯ


# насклько значение отклняется от среднего z = (x - среднее) / стандартное отклонение.
# в диапазон. 0-1 u = (x - минимум) / (максимум - минимум).

scaled = data.copy()
parameters = {}

for c in FEATURES:
    low = data[c].min()
    high = data[c].max()

    scaled[c + "_z"] = (data[c] - mu[c]) / sigma[c]
    scaled[c + "_minmax"] = (data[c] - low) / (high - low)

    parameters[c] = {
        "mean": float(mu[c]),
        "std_ddof0": float(sigma[c]),
        "min": float(low),
        "max": float(high),
    }

#  log10(MolWt / (1 г/моль)).
scaled["MolWt_log10"] = np.log10(data["MolWt"])

cols = [
    "LogS_z",
    "MolWt_z",
    "LogS_minmax",
    "MolWt_minmax",
]

display(
    pd.DataFrame(
        {
            "Среднее": scaled[cols].mean(),
            "Стандартное отклонение (N)": scaled[cols].std(ddof=0),
            "Минимум": scaled[cols].min(),
            "Максимум": scaled[cols].max(),
        }
    ).round(6)
)

scaled.to_csv(OUT / "aqsoldb_scaled.csv", index=False)

(OUT / "scaling_parameters.json").write_text(
    json.dumps(parameters, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(
    x=data["MolWt"],
    bins="fd",
    stat="density",
    ax=axes[0],
)

axes[0].set(
    xlabel=labels["MolWt"],
    ylabel="Плотность",
    title="Исходная масса",
)

sns.histplot(
    x=scaled["MolWt_log10"],
    bins="fd",
    stat="density",
    ax=axes[1],
)

axes[1].set(
    xlabel="log₁₀(MolWt / (1 г/моль))",
    ylabel="Плотность",
    title="После логарифмирования",
)

plt.tight_layout()
plt.savefig(
    OUT / "02_mass_transform.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print(
    "Асимметрия MolWt до и после логарифмирования:",
    round(float(skew(data["MolWt"])), 3),
    round(float(skew(scaled["MolWt_log10"])), 3),
)

<a id="section-6"></a>
## 6. Генеральные и выборочные характеристики

Формируются случайные выборки с возвращением и без возвращения. Сопоставляются средние и дисперсии выборок с характеристиками генеральной совокупности, а также вычисляется стандартная ошибка среднего. Для неё используется поправка на конечную совокупность (FPC):

\[
SE(\bar X)=\sqrt{\frac{\sigma^2}{n}\,fpc},\qquad
fpc=\begin{cases}
1,&\text{при отборе с возвращением},\\[4pt]
\frac{N-n}{N-1},&\text{при отборе без возвращения}.
\end{cases}
\]

[К оглавлению](#contents)

In [ ]:
#@title 6. Генеральные и выборочные характеристики {display-mode: "form"}


# 6. ГЕНЕРАЛЬНЫЕ И ВЫБОРОЧНЫЕ ХАРАКТЕРИСТИКИ




rng = np.random.default_rng(SEED)
n_sample = min(500, N)

comparison_rows = []

for replace, method, suffix in [
    (True, "С возвращением", "with"), #могут повторяться
    (False, "Без возвращения", "without"), #ровн 500
]:
    indices = rng.choice(N, size=n_sample, replace=replace)
    sample = data.iloc[indices].copy()

    sample.to_csv(
        OUT / f"sample_{suffix}.csv",
        index=False,
    )

    for c in FEATURES:
        mean_hat = sample[c].mean()
        var_hat = sample[c].var(ddof=1)

        if not replace:
            var_hat *= (N - 1) / N

        # Поправка на конечную совокупность
        fpc = (
            1.0
            if replace
            else (N - n_sample) / (N - 1)
        )

        comparison_rows.append(
            {
                "Отбор": method,
                "Признак": c,
                "n": n_sample,
                "Среднее базы": mu[c],
                "Среднее выборки": mean_hat,
                "Абс. ошибка среднего": abs(mean_hat - mu[c]),
                "Дисперсия базы": variance[c],
                "Оценка дисперсии": var_hat,
                "Стандартная ошибка среднего": np.sqrt(
                    variance[c] / n_sample * fpc
                ),
                "Уникальных строк": len(np.unique(indices)),
            }
        )

comparison = pd.DataFrame(comparison_rows)

display(comparison.round(5))
comparison.to_csv(
    OUT / "sample_comparison.csv",
    index=False,
)



<a id="section-7"></a>
## 7. Сходимость статистических оценок

Для набора объёмов выборки выполняется серия повторных отборов. Качество оценок среднего и дисперсии характеризуется RMSE:

\[
RMSE(\hat\theta)=
\sqrt{\frac{1}{B}\sum_{b=1}^{B}
(\hat\theta_b-\theta)^2}.
\]

На графиках сходимости обе оси логарифмические. Относительные ошибки \(RMSE/\sigma\) и \(RMSE/\sigma^2\), а также теоретическая величина \(\sqrt{fpc/n}\) являются безразмерными.

[К оглавлению](#contents)

In [ ]:
#@title 7. Сходимость статистических оценок {display-mode: "form"}

# 7. СХОДИМОСТЬ ОЦЕНОК


# Для каждого объёма n выполняется 200 повторных отборов

B = 200

sizes = sorted(
    set(
        n
        for n in [10, 30, 100, 300, 1000, 3000, 8000, N]
        if 2 <= n <= N
    )
)

values = data[FEATURES].to_numpy(dtype=float)

true_mean = values.mean(axis=0)
true_var = values.var(axis=0, ddof=0)
true_std = np.sqrt(true_var)

rng_mc = np.random.default_rng(SEED + 1)
records = []

for replace, method in [
    (True, "С возвращением"),
    (False, "Без возвращения"),
]:
    for n in sizes:
        means = np.empty((B, len(FEATURES)))
        variances = np.empty_like(means)

        for b in range(B):
            indices = rng_mc.choice(
                N,
                size=n,
                replace=replace,
            )

            selected = values[indices]

            means[b] = selected.mean(axis=0)
            variances[b] = selected.var(axis=0, ddof=1)

        if not replace:
            variances *= (N - 1) / N

        mean_rmse = np.sqrt(
            ((means - true_mean) ** 2).mean(axis=0)
        )

        var_rmse = np.sqrt(
            ((variances - true_var) ** 2).mean(axis=0)
        )

        fpc = (
            1.0
            if replace
            else (N - n) / (N - 1)
        )

        for j, c in enumerate(FEATURES):
            records.append(
                {
                    "Отбор": method,
                    "Признак": c,
                    "n": n,
                    "RMSE среднего": mean_rmse[j],
                    "RMSE среднего / σ": (
                        mean_rmse[j] / true_std[j]
                    ),
                    "Теория RMSE среднего / σ": np.sqrt(fpc / n),
                    "RMSE дисперсии": var_rmse[j],
                    "RMSE дисперсии / σ²": (
                        var_rmse[j] / true_var[j]
                    ),
                    "Доля |ошибки среднего| > 0.25σ": float(
                        (
                            abs(means[:, j] - true_mean[j])
                            > 0.25 * true_std[j]
                        ).mean()
                    ),
                }
            )

convergence = pd.DataFrame(records)

display(convergence.round(6))
convergence.to_csv(
    OUT / "convergence.csv",
    index=False,
)


# графики сходимости
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for j, method in enumerate(
    ["С возвращением", "Без возвращения"]
):
    subset = convergence[
        convergence["Отбор"] == method
    ]

    #  n=N без возвращения ошибки близки к 0

    plotted = (
        subset
        if method == "С возвращением"
        else subset[subset["n"] < N]
    )

    for c in FEATURES:
        p = plotted[plotted["Признак"] == c]

        axes[0, j].loglog(
            p["n"],
            p["RMSE среднего / σ"],
            "o-",
            label=c,
        )

        axes[1, j].loglog(
            p["n"],
            p["RMSE дисперсии / σ²"],
            "o-",
            label=c,
        )

    theory = plotted[
        plotted["Признак"] == FEATURES[0]
    ]

    axes[0, j].loglog(
        theory["n"],
        theory["Теория RMSE среднего / σ"],
        "--",
        color="black",
        label="Теория",
    )

    for row, ylabel in [
        (0, "RMSE среднего / σ"),
        (1, "RMSE дисперсии / σ²"),
    ]:
        axes[row, j].set(
            xlabel="Объём выборки n",
            ylabel=ylabel,
            title=method,
        )

        axes[row, j].legend()

plt.tight_layout()
plt.savefig(
    OUT / "03_convergence.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)






<a id="section-8"></a>
## 8. Выводы по основному исследованию

На основе рассчитанных характеристик автоматически формируются итоговые численные выводы по размеру очищенной совокупности, описательным статистикам, отмеченным IQR-наблюдениям и сходимости оценок.

[К оглавлению](#contents)

In [ ]:
#@title 8. Формирование выводов {display-mode: "form"}

# 8. ВЫВОДЫ С РАССЧИТАННЫМИ ЧИСЛАМИ


conclusions = [
    (
        f"Генеральная совокупность исследования: "
        f"{N} соединений AqSolDB."
    ),
    (
        f"Полных повторов: {duplicate_count}; "
        f"некорректных строк: {invalid_count}."
    ),
]

for c in FEATURES:
    conclusions.append(
        f"{c}: среднее {mu[c]:.5f}; "
        f"медиана {data[c].median():.5f}; "
        f"дисперсия (делитель N) {variance[c]:.5f}; "
        f"асимметрия {summary.loc[c, 'Асимметрия']:.3f}; "
        f"по IQR отмечено "
        f"{int(outliers.loc[c, 'Отмечено'])} наблюдений, "
        f"они сохранены."
    )

    p = convergence[
        (convergence["Отбор"] == "С возвращением")
        & (convergence["Признак"] == c)
    ].sort_values("n")

    first = p.iloc[0]
    last = p.iloc[-1]

    conclusions.append(
        f"{c}: при n={int(first['n'])} "
        f"и n={int(last['n'])} "
        f"RMSE среднего/σ: "
        f"{first['RMSE среднего / σ']:.4f} и "
        f"{last['RMSE среднего / σ']:.4f}; "
        f"RMSE дисперсии/σ²: "
        f"{first['RMSE дисперсии / σ²']:.4f} и "
        f"{last['RMSE дисперсии / σ²']:.4f}."
    )


text = "\n\n".join(
    f"{i}. {t}"
    for i, t in enumerate(conclusions, 1)
)

display(Markdown(text))

(OUT / "conclusions.md").write_text(
    text,
    encoding="utf-8",
)


# сохранение


archive = Path("Lab01_AqSolDB_Daria_results.zip")

with zipfile.ZipFile(
    archive,
    "w",
    zipfile.ZIP_DEFLATED,
) as bundle:
    for item in sorted(OUT.iterdir()):
        if item.is_file():
            bundle.write(item, arcname=item.name)

print("Результаты сохранены в", archive)

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))

<a id="section-9"></a>
## 9. Дополнительный анализ признака MolLogP

Дополнительный блок рассматривает распределение `MolLogP`, выборочные средние, корреляции и сопоставление эмпирического распределения с нормальным законом. Для него отдельно загружается набор `curated-solubility-dataset.csv` из репозитория `dmol-book`; поэтому результаты этого раздела рассматриваются отдельно и не объединяются с очищенной совокупностью AqSolDB из разделов 1–8.

**Примечание по единицам и шкалам:** `MolLogP` — логарифмический показатель и безразмерен; `Solubility` и `MolWt` отображаются в исходных единицах набора данных. Для матриц корреляций шкала коэффициента ограничена интервалом \([-1,1]\).

[К оглавлению](#contents)

In [ ]:
#@title 9. Дополнительный анализ MolLogP {display-mode: "form"}
# Описание, EDA, распределение
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew
import scipy.stats as stats
import seaborn as sns

# Единая стилистика графиков исследования
PLOT_COLORS = {
    "primary": "#3366A6",
    "secondary": "#E07A3F",
    "tertiary": "#3A8D75",
    "neutral": "#555555",
    "grid": "#D9DEE7",
}
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette=[PLOT_COLORS["primary"], PLOT_COLORS["secondary"],
             PLOT_COLORS["tertiary"], PLOT_COLORS["neutral"]],
    font="DejaVu Sans",
    rc={
        "figure.figsize": (10, 6),
        "figure.dpi": 110,
        "savefig.dpi": 180,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "axes.edgecolor": "#AAB2BF",
        "axes.linewidth": 0.8,
        "grid.color": PLOT_COLORS["grid"],
        "grid.alpha": 0.7,
        "grid.linewidth": 0.7,
        "axes.spines.top": False,
        "axes.spines.right": False,
    },
)


# Загрузка генеральной совокупности данных
url = 'https://raw.githubusercontent.com/whitead/dmol-book/main/data/curated-solubility-dataset.csv'
df = pd.read_csv(url)


features = ['MolLogP', 'Solubility', 'MolWt']
data = df[features].dropna().reset_index(drop=True)

population = data['MolLogP']
print(f'Размер генеральной совокупности по MolLogP: {population.count()}')

# Расчет основных статистик генеральной совокупности
mu = population.mean()
variance = population.var(ddof=0)  # Делитель N
sigma = np.sqrt(variance)

summary = pd.DataFrame(
    {
        'N': [population.count()],
        'Среднее': [mu],
        'Медиана': [population.median()],
        'Дисперсия (N)': [variance],
        'Стандартное отклонение': [sigma],
        'Минимум': [population.min()],
        'Q1': [population.quantile(0.25)],
        'Q3': [population.quantile(0.75)],
        'Максимум': [population.max()],
        'Асимметрия': [skew(population, bias=True)],
        'Эксцесс': [kurtosis(population, fisher=True, bias=True)],
    },
    index=['MolLogP'],
)

display(summary.round(5))

# Визуализация распределения MolLogP (гистограмма + KDE и Boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    population,
    kde=True,
    ax=axes[0],
    color=PLOT_COLORS["primary"],
    bins=50,
    stat='density',
)
axes[0].set_title('Гистограмма и KDE плотности: MolLogP')
axes[0].set_xlabel('MolLogP')
axes[0].set_ylabel('Плотность вероятности')

sns.boxplot(y=population, ax=axes[1], color=PLOT_COLORS["secondary"])
axes[1].set_title('Boxplot (оценка выбросов): MolLogP')
axes[1].set_ylabel('MolLogP')

plt.tight_layout()
plt.show()

# Исследование сходимости выборочных оценок
sample_sizes = [10, 50, 100, 500, 1000, 5000]
est_results = []

np.random.seed(42)
for n in sample_sizes:
  s_without = population.sample(n=n, replace=False)
  s_with = population.sample(n=n, replace=True)

  est_results.append({
      'N': n,
      'Mean_without': s_without.mean(),
      'Error_without': abs(s_without.mean() - mu),
      'Mean_with': s_with.mean(),
      'Error_with': abs(s_with.mean() - mu),
  })

df_est = pd.DataFrame(est_results)
print('Сходимость выборочного среднего (с возвращением и без):')
print(df_est.to_string(index=False))





# Матрицы корреляций Пирсона и Спирмана рассчитываются и визуализируются ниже.

corr_val, p_value = stats.pearsonr(data['Solubility'], data['MolLogP'])
print(f'\nСтатистическая гипотеза о связи Solubility и MolLogP:')
print(f'Коэффициент корреляции: {corr_val:.4f}, p-value: {p_value:.4e}')

if p_value < 0.05:
  print(
      'Результат: Отвергаем H0.'
  )
else:
  print('Не отвергаем НО')

fit_mu, fit_std = stats.norm.fit(population)
print(f'Параметры нормального распределения: μ = {fit_mu:.4f}, σ = {fit_std:.4f}')

# Графическое сравнение с нормальным законом
plt.figure(figsize=(10, 6))
count, bins, ignored = plt.hist(
    population,
    bins=50,
    density=True,
    alpha=0.6,
    color=PLOT_COLORS["primary"],
    label='Эмпирическая плотность',
    )
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, fit_mu, fit_std)
plt.plot(
    x,
    p,
    '-',
    color=PLOT_COLORS["secondary"],
    linewidth=2.2,
    label=f'Теоретическая нормальная кривая\n(μ={fit_mu:.2f}, σ={fit_std:.2f})',
)

plt.title('Математическое описание распределения MolLogP')
plt.xlabel('MolLogP')
plt.ylabel('Плотность вероятности')
plt.legend()
plt.show()

corr_pearson = data.corr(method='pearson')
corr_spearman = data.corr(method='spearman')

print('Матрица корреляций Пирсона')
display(corr_pearson.round(4))

print('\n Матрица корреляций Спирмана')
display(corr_spearman.round(4))

# Визуализация матриц корреляций
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    corr_pearson,
    annot=True,
    cmap='vlag',
    vmin=-1,
    vmax=1,
    fmt='.2f',
    ax=axes[0],
)
axes[0].set_title('Матрица корреляции Пирсона ')

sns.heatmap(
    corr_spearman,
    annot=True,
    cmap='vlag',
    vmin=-1,
    vmax=1,
    fmt='.2f',
    ax=axes[1],
)
axes[1].set_title('Матрица корреляции Спирмана ')

plt.tight_layout()
plt.show()



